# CONTENT-BASED FILTERING: A SIMPLE EXAMPLE

Adapted from https://heartbeat.fritz.ai/recommender-systems-with-python-part-i-content-based-filtering-5df4940bd831.

# Preparation

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel 

dataset = pd.read_csv("products.csv")

In [2]:
dataset.head()

,id,description
0,1,"Alpine guide pants - Skin in, climb ice, switc..."
1,2,"Alpine wind jkt - On high ridges, steep ice an..."
2,3,Ascensionist jkt - Our most technical soft she...
3,4,"Atom - A multitasker's cloud nine, the Atom pl..."
4,5,Print banded betina btm - Our fullest coverage...


In [3]:
dataset.tail()

,id,description
481,482,Cap 2 bottoms - Cut loose from the maddening c...
482,483,Cap 2 crew - This crew takes the edge off fick...
483,484,All-time shell - No need to use that morning T...
484,485,All-wear cargo shorts - All-Wear Cargo Shorts ...
485,486,All-wear shorts - Time to simplify? Our All-We...


# TF-IDF Representation 

In [4]:
tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 3), min_df=0, stop_words='english')
tfidf_matrix = tf.fit_transform(dataset['description'])
print(tfidf_matrix)

  (0, 4937)	0.02448244315422262
  (0, 30182)	0.009560170095611257
  (0, 530)	0.050750888763709104
  (0, 1322)	0.06108267825208003
  (0, 48773)	0.06108267825208003
  (0, 5179)	0.06108267825208003
  (0, 4938)	0.009560170095611257
  (0, 15672)	0.028788088733595056
  (0, 12869)	0.037713621972009004
  (0, 11054)	0.032013278376321314
  (0, 41055)	0.044232255367859054
  (0, 34833)	0.05726952215955915
  (0, 1177)	0.05726952215955915
  (0, 1563)	0.05246551362583484
  (0, 11078)	0.05246551362583484
  (0, 1544)	0.05726952215955915
  (0, 30159)	0.05726952215955915
  (0, 14577)	0.04930119409900137
  (0, 4970)	0.012673580815654777
  (0, 4927)	0.009404400073566046
  (0, 45824)	0.009404400073566046
  (0, 24949)	0.009404400073566046
  (0, 26474)	0.05456404485622998
  (0, 44614)	0.05456404485622998
  (0, 37709)	0.06108267825208003
  :	:
  (485, 45825)	0.012738055846000244
  (485, 11403)	0.012738055846000244
  (485, 4943)	0.012738055846000244
  (485, 4922)	0.050952223384000975
  (485, 33370)	0.0344063467

![tf-idf](tfidf.png "tf-idf")

# Cosine Simlarities

Compute cosine simlarities between each pair of product descriptions.

In [5]:
cosine_similarities = linear_kernel(tfidf_matrix, tfidf_matrix) 

For each item, store the list of most similar items to it in results, sorted by similarity starting with the most smilar. 

In [6]:
results = {}
for idx, row in dataset.iterrows():
   similar_indices = cosine_similarities[idx].argsort()[:-100:-1] 
   similar_items = [(cosine_similarities[idx][i], dataset['id'][i]) for i in similar_indices] 
   results[row['id']] = similar_items[1:]

Recomendation function

In [7]:
# Return the first bit of the description of item id
def item(id):
  return dataset.loc[dataset['id'] == id]['description'].tolist()[0].split(' - ')[0] 

# Just reads the results out of the dictionary.
def recommend(item_id, num):
    print("Recommending " + str(num) + " products similar to " + item(item_id) + "...")   
    print("-------")    
    recs = results[item_id][:num]   
    for rec in recs: 
       print("Recommended: " + item(rec[1]) + " (id: " + str(rec[1]) + ", score:" +      str(rec[0]) + ")")

# Example use

In [8]:
recommend(200,5)

Recommending 5 products similar to Micro puff vest...
-------
Recommended: Micro puff vest (id: 407, score:0.8942380461378928)
Recommended: Down sweater (id: 175, score:0.2359741907891981)
Recommended: Down sweater vest (id: 31, score:0.2243994578267621)
Recommended: Down sweater (id: 461, score:0.2227734726615042)
Recommended: Down sweater vest (id: 276, score:0.20728520074319812)
